# AMFI Monthly Flow Report Layout & Target Availability Inspection

**Project:** FundFlow Sentinel  
**Artifact:** `notebooks/00_source_inspection.ipynb`  
**Owner:** Agent A (Data & Ingestion Lead)  
**Date:** 2026-10-28  

---

## 1. Executive Summary & Verification Goals

Before building an AMFI report ingestion pipeline, we must inspect official AMFI monthly reports across historical periods (FY2018-19 to FY2026-27) to ground all parser assumptions in empirical fact:
1. **Granularity Check:** Verify whether bulk public reports provide scheme-level or category-level gross redemption. (Design assumption: category-level only).
2. **Target Field Identification:** Identify the exact gross redemption column and distinguish it from net inflow/outflow.
3. **Switch-Out Treatment:** Confirm whether repurchase/redemption includes switch-outs.
4. **Layout Evolution & Structural Breaks:** Trace file formats (BIFF8 vs OOXML), sheet naming changes, and structural events (SEBI recategorization, Flexi Cap introduction).
5. **Publication Lag Evidence:** Document HTTP `Last-Modified` headers and in-sheet release dates to determine the non-leakage availability boundary (`available_from`).
6. **Category Mapping:** Propose exact `category_map.csv` mappings with validity date ranges.

In [1]:
import io
import json
from pathlib import Path
import pandas as pd

# Load empirical inspection findings recorded by scripts/inspect_amfi_reports.py
evidence_file = Path("../docs/amfi_inspection_evidence.json")
if not evidence_file.exists():
    evidence_file = Path("docs/amfi_inspection_evidence.json")

with open(evidence_file, "r") as f:
    evidence_data = json.load(f)

summary_rows = []
for item in evidence_data:
    summary_rows.append({
        "Period": item["period"],
        "Format": "OOXML (.xlsx)" if "OOXML" in item["file_type"] else "BIFF8 (.xls)",
        "HTTP Last-Modified": item["http_last_modified"],
        "In-Sheet Release Date": item.get("in_file_release") or "Not printed",
        "Sheets": ", ".join(item["sheets"]),
        "Units": item["units"],
        "Categories Present": len(item["sample_categories"]),
        "Notes": item["notes"]
    })

df_summary = pd.DataFrame(summary_rows)
df_summary

## 2. Granularity: Scheme vs Category Level

**Inspection Evidence:**
- In all monthly bulk reports (e.g., `amsep2024repo.xls`, `amsep2026repo.xls`, `amapr2019repo.xls`), the primary data sheet contains rows corresponding strictly to **SEBI Mutual Fund Categories** (such as *Large Cap Fund*, *Mid Cap Fund*, *Small Cap Fund*, *Flexi Cap Fund*, *Corporate Bond Fund*, etc.).
- The column titled `Scheme Name` or `Scheme Category` actually lists mutual fund **categories**, not individual funds.
- Individual schemes appear nowhere in the monthly flow tables; instead, a column titled `No. of Schemes` provides the count of schemes operating within that category.
- Sheet 2 (`New Schemes` or `NSR`) reports newly launched fund offers (NFOs) only, without historical or subsequent redemption series.

**Conclusion:**  
Public AMFI bulk monthly reports provide **category-level data only**. Scheme-level gross redemption is not available in free bulk reports. The architectural decision to model flows at the category-month level is strictly verified.

## 3. Definitions, Flow Arithmetic & Switch-Outs

### Column Definitions
1. **Gross Inflows (`gross_inflow_cr`):**  
   Report column: `Funds Mobilized for the month of <Month> <Year> (INR in crore)`.  
   Includes new sales, SIP contributions, NFO purchases, and switch-ins.
2. **Gross Redemptions (`gross_redemption_cr`):**  
   Report column: `Repurchase/ Redemption for the month of <Month> <Year> (INR in crore)`.  
   Includes all investor unit redemptions and switch-outs.
3. **Net Flow (`net_flow_cr`):**  
   Report column: `Net Inflow (+ve)/Outflow (-ve) for the month of <Month> <Year> (INR in crore)`.  
   Identity: `Net Flow = Funds Mobilized - Repurchase/Redemption`.
4. **Closing AUM (`aum_close_cr`):**  
   Report column: `Net Assets Under Management as on <End Date> (INR in crore)`.
5. **Average AUM (`aaum_cr`):**  
   Report column: `Average Net Assets Under Management for the month <Month> <Year> (INR in crore)`.

### Switch-Outs & Liquid/Overnight Distortion
- In mutual fund accounting guidelines in India, a switch-out between schemes within an AMC is legally and computationally processed as a redemption from the origin scheme and a purchase into the destination scheme.
- In cash-management categories (Overnight and Liquid Funds), corporate treasuries execute massive short-term churn. For example, in September 2024, Overnight Funds recorded ₹5,25,369 cr in inflows and ₹5,44,732 cr in redemptions on an AUM base of ₹67,274 cr (redemption rate > 800% in a single month).
- For equity and hybrid categories, typical redemption rates are 1%–4% of AUM.
- **Decision:** Liquid and Overnight categories must be strictly excluded from pooled predictive models to avoid severe distribution distortion.

## 4. Publication Lag and the `available_from` Timestamp Rule

Empirical evidence collected across 8 historical checkpoints:
- **September 2018:** Released `08.10.2018` (HTTP header: 11 Oct 2018)
- **April 2019:** Released `09-May-2019` (HTTP header: 10 Jun 2019)
- **September 2020:** Released `08-Oct-2020` (HTTP header: 08 Oct 2020)
- **January 2021:** Released `09-Feb-2021` (HTTP header: 09 Feb 2021)
- **September 2022:** Released `10-Oct-2022` (HTTP header: 10 Oct 2022)
- **September 2024:** Released `10-Oct-2024` (HTTP header: 10 Oct 2024)
- **September 2026:** HTTP header: `12 Oct 2026`

**Timestamp Rule:**  
AMFI reports are consistently published between the 8th and 12th of the following month (and up to the 24th for April reports due to financial year-end accounting). Assuming month-end availability (e.g. 30 September for September data) introduces look-ahead bias. Any dataset row for month `YYYY-MM` must carry `available_from` set to the verified publication timestamp in ISO UTC format (e.g., `YYYY-(MM+1)-10T12:00:00Z`).

## 5. Structural Breaks and Layout Evolution

1. **FY2018-19 (BIFF8 Multi-Sheet):** Reports used classic BIFF8 `.xls` files with 4 sheets (`a+s+r`, `sales`, `red+assets`, `AAUM`). Data was organized across separate tables.
2. **FY2019-20 to FY2024-25 (OOXML Single-Sheet):** Reports migrated to OOXML (`.xlsx` formatted with `.xls` file extension). Consolidated into a single main sheet named `<Month> <YY>` plus a secondary `New Schemes` sheet.
3. **Flexi Cap Structural Break (January 2021):** SEBI introduced the `Flexi Cap Fund` category in late 2020. In January 2021, 16 existing Multi Cap funds converted to Flexi Cap. AMFI explicitly footnoted: *"During Jan 2021, 16 Multi Cap Funds were re-categorized as Flexi Cap Fund... resulting in negative funds mobilized and redemption figures under Multi Cap Fund."* Flexi Cap history begins at `2021-01`.
4. **FY2025-26+ (BIFF8 Return):** Files returned to BIFF8 `.xls` formatting, using standardized sheet names `MCR_MonthlyReport` and `NSR`.

**Parser Requirement:** The ingestion parser must dynamically detect file format (magic bytes `504b0304` for OOXML vs `d0cf11e0` for BIFF8) and apply layout adapters rather than assuming a single file type or sheet name.

In [2]:
# Verification of Category Map proposal
category_map_proposal = [
    {"raw_category": "Large Cap Fund", "category": "Large Cap", "valid_from": "2018-04", "valid_to": "9999-12"},
    {"raw_category": "Mid Cap Fund", "category": "Mid Cap", "valid_from": "2018-04", "valid_to": "9999-12"},
    {"raw_category": "Small Cap Fund", "category": "Small Cap", "valid_from": "2018-04", "valid_to": "9999-12"},
    {"raw_category": "Flexi Cap Fund", "category": "Flexi Cap", "valid_from": "2021-01", "valid_to": "9999-12"},
    {"raw_category": "Balanced Hybrid Fund/Aggressive Hybrid Fund", "category": "Aggressive Hybrid", "valid_from": "2018-04", "valid_to": "9999-12"},
    {"raw_category": "Corporate Bond Fund", "category": "Corporate Bond", "valid_from": "2018-04", "valid_to": "9999-12"},
    {"raw_category": "Liquid Fund", "category": "Liquid", "valid_from": "2018-04", "valid_to": "9999-12"},
    {"raw_category": "Overnight Fund", "category": "Overnight", "valid_from": "2018-04", "valid_to": "9999-12"}
]
df_cat_map = pd.DataFrame(category_map_proposal)
df_cat_map